# QCoDeS Example with QuTech M1b Preamplifier

This notebook explains how the QuTech M1b current preamplifier module works and shows the main features of its QCoDeS driver.

The M1b is a low-noise current-to-voltage converter (transimpedance amplifier) designed for use in the IVVI rack system. It converts small currents (nA to pA range) into measurable voltages.

**Documentation:** https://qtwork.tudelft.nl/~schouten/ivvi/doc-mod/docm1b.htm

## Virtual Driver

This is a **virtual driver** - it does not communicate with the physical instrument. It is the user's responsibility to manually set the physical instrument to match the driver settings. The driver helps track settings and calculate derived quantities like total gain and input resistance.

## Quick Start: Measurement Example

### Import Required Libraries

In [1]:
from qcodes_contrib_drivers.drivers.QuTech.M1b import M1b, CurrentParameter

### Create an M1b Instance

In [2]:
preamp = M1b("m1b_preamp")

### Setting Module Slot

The M1b module can be placed in slot Ma (top, iso-out 1) or Mb (bottom, iso-out 2):

In [3]:
preamp.slot("Ma")

### Setting Transimpedance Gain

The main gain setting converts current to voltage. Choose from: "1M", "10M", "100M", "1G"

In [4]:
# Set gain to 100 MΩ (100 MV/A)
preamp.gain("100M")

### Setting Postgain

The postgain multiplies the output. Options: "x1", "x100ac", "x100dc"

In [5]:
# Set to x1 for DC measurements (no postgain)
preamp.postgain("x1")

### Using CurrentParameter for Measurements

The `CurrentParameter` class automatically converts voltage measurements to current using the M1b's total gain.

In [6]:
from qcodes.instrument_drivers.mock_instruments import DummyInstrument

# Create a mock DMM (digital multimeter) that simulates measuring the M1b output
# In a real setup, this would be an actual DMM like Keysight 34465A
dmm = DummyInstrument(name="dmm", gates=["voltage"])

# Create CurrentParameter that links voltage measurement to M1b
current_param = CurrentParameter(
    measured_param=dmm.voltage,  # Use the DMM voltage parameter
    current_amplifier_instrument=preamp,
    name="current"
)

In [7]:
# Create a mock gate voltage source
gate_dac = DummyInstrument(name="gate_dac", gates=["voltage"])

# Simulate a simple I-V sweep
print("Gate voltage sweep with current measurement:")
print("\nGate Voltage (V) | M1b Output (mV) | Current (nA)")
print("-" * 55)

for v_gate in [-0.1, -0.05, 0.0, 0.05, 0.1]:
    gate_dac.voltage(v_gate)
    
    # Simulate M1b output voltage that varies with gate voltage
    # In reality, this would be measured by your DMM
    dmm.voltage(0.05 + 0.5 * v_gate)  # Simulated response
    
    # Get current measurement (automatically calculated from voltage)
    voltage_raw, current = current_param.get()
    
    print(f"{v_gate:+15.3f} | {voltage_raw*1e3:15.1f} | {current*1e9:12.3f}")

print("\nMeasurement complete!")

Gate voltage sweep with current measurement:

Gate Voltage (V) | M1b Output (mV) | Current (nA)
-------------------------------------------------------
         -0.100 |             0.0 |        0.000
         -0.050 |            25.0 |        0.250
         +0.000 |            50.0 |        0.500
         +0.050 |            75.0 |        0.750
         +0.100 |           100.0 |        1.000

Measurement complete!


### Additional parameters

In [8]:
# Mute the output
preamp.muted(True)

# Unmute
preamp.muted(False)

### Output Controls

In [9]:
# Reference to ground (default)
preamp.reference("ground")

# Reference to external ref-in (ideally a cold ground for cryogenic measurements)
preamp.reference("ref-in")

### Reference Settings

In [10]:
# Low Rin mode - lower input resistance, faster response
preamp.input_resistance_setting("Low Rin")

# Low Noise mode - higher input resistance, lower noise (default)
preamp.input_resistance_setting("Low Noise")

# Get calculated input resistance
rin = preamp.input_resistance()
print(f"Input resistance: {rin:.2e} Ω")
print(f"Input resistance: {rin/1e3:.1f} kΩ")

Input resistance: 1.02e+05 Ω
Input resistance: 102.0 kΩ


### Input Resistance Settings

In [11]:
# The total gain is the product of the base gain and postgain
preamp.postgain("x1")  # Reset to x1 for this example
total_gain = preamp.total_gain()
print(f"Total gain: {total_gain:.2e} V/A")
print(f"Total gain: {total_gain/1e9:.0f} GV/A")

Total gain: 1.00e+08 V/A
Total gain: 0 GV/A


### Total Gain Calculation

In [12]:
# x1: No postgain (direct output)
preamp.postgain("x1")

# x100ac: AC-coupled 100x postgain (useful for noise measurements with DC offset)
preamp.postgain("x100ac")

# x100dc: DC-coupled 100x postgain (maximum total gain: 100 GV/A)
preamp.postgain("x100dc")

## Important Notes

1. **Virtual Driver**: This driver does not communicate with the hardware. Always ensure the physical M1b settings match the driver settings.

2. **Gain Settings**: Choose gain to avoid saturation (±10V output limit) while maintaining good signal-to-noise ratio.

3. **Bandwidth**: Higher gains have lower bandwidth. Check M1b documentation for bandwidth specifications.

4. **Input Protection**: The M1b has input protection, but avoid exceeding specified current ranges.

5. **Reference**: Use external ref-in connected to a cold ground for lowest noise in cryogenic measurements.

6. **AC Coupling**: When using x100ac postgain, the AC coupling has a cutoff frequency - check documentation for your measurement frequency range.

## Cleanup

In [13]:
# Clean up mock instruments
gate_dac.close()
dmm.close()
# Close the instrument when done
preamp.close()